# MCNP Radiation Transport Example

This notebook demonstrates how to use the RadSim MCNP API to simulate radiation transport. We'll walk through:

1. Setting up the Java environment
2. Creating radiation sources
3. Defining geometries for sources and shielding
4. Running MCNP simulations
5. Analyzing and visualizing results

This example simulates a Cs-137 source with both beta and gamma emissions, transported through a shielding geometry.

## 1. Set Up Java Environment

First, we need to initialize the JVM and import the necessary Java classes.

In [ ]:
# Initialize the JVM and import required libraries
import startJVM
import jpype
import jpype.imports

In [ ]:
# Import necessary Java classes
import numpy as np
import matplotlib.pyplot as plt

from java.nio.file import Path
from java.util import ArrayList

from gov.llnl.ensdf.decay import BNLDecayLibrary
from gov.llnl.rtk.data import EnergyScaleFactory
from gov.llnl.rtk.physics import EmissionCalculator, SourceImpl, Nuclides
from gov.llnl.rtk.physics import SphericalModel, LayerImpl, MaterialImpl
from gov.llnl.rtk.physics import SphericalSection, CylindricalSection, ConicalSection
from gov.llnl.rtk.flux import FluxBinned, FluxGroupBin
from gov.llnl.rtk.flux import FluxTrapezoid, FluxGroupTrapezoid
from gov.llnl.rtk.flux import FluxSpectrum, FluxLineStep
from gov.nist.physics.xray import NISTXrayLibrary

# Import the MCNP classes
from gov.llnl.rtk.mcnp import RadSim_MCNP_Job, MCNP_Job
from gov.llnl.rtk.mcnp import MCNP_Photon, MCNP_Electron
from gov.llnl.math.euclidean import Vector3

# Import helper functions
from geometry import get_offset_vector

## 2. Helper Functions

Let's define some helper functions for working with beta spectra and gamma lines.

In [ ]:
def parse_beta_file(path):
    """Parse a beta spectrum file into a FluxTrapezoid object."""
    with open(path, 'r') as f:
        lines = f.readlines()

    energies = []
    densities = []
    for line in lines:
        entries = line.split()
        try:
            energies.append(float(entries[0]))
            densities.append(float(entries[1]))
        except Exception:
            pass

    flux = FluxTrapezoid()
    for i in range(1, len(energies)):
        flux.addPhotonGroup(FluxGroupTrapezoid(energies[i-1], energies[i], densities[i-1], densities[i]))
    return flux


def get_flux_from_lines(isotopes, activities):
    """Create a flux object from isotopes and their activities."""
    # Load the decay library
    decay_lib = BNLDecayLibrary()
    decay_lib.setXrayLibrary(NISTXrayLibrary.getInstance())
    decay_lib.loadFile(Path.of("../data/BNL2023.txt"))

    # Set up the emission calculator
    calculator = EmissionCalculator()
    calculator.setDecayLibrary(decay_lib)

    # Add sources
    sources = ArrayList()
    for isotope, activity in zip(isotopes, activities):
        # Using SourceImpl.fromActivity consistent with sourcegen_example
        sources.add(SourceImpl.fromActivity(Nuclides.get(isotope), activity))

    # Calculate the emissions consistently with sourcegen_example
    emissions = calculator.apply(sources)

    # Create the flux
    flux = FluxBinned()
    for gamma in emissions.getGammas():
        flux.addPhotonLine(FluxLineStep(gamma.getEnergy().getValue(), gamma.getIntensity().getValue(), 0.0))

    return flux


def add_fluxes(fluxes):
    """Combine multiple flux objects into one."""
    # Get the bins from the first flux
    lower_bins = []
    upper_bins = []
    for group in fluxes[0].getPhotonGroups():
        lower_bins.append(group.getEnergyLower())
        upper_bins.append(group.getEnergyUpper())

    # Sum the counts
    total_counts = []
    for flux in fluxes:
        groups = flux.getPhotonGroups()
        counts = []
        for group in groups:
            counts.append(group.getCounts())
        total_counts.append(counts)
    total_counts = np.array(total_counts).sum(axis=0)

    # Create a new flux with summed counts
    total_flux = FluxBinned()
    for lower, upper, counts in zip(lower_bins, upper_bins, total_counts):
        total_flux.addPhotonGroup(FluxGroupBin(lower, upper, counts, 0.0))
    return total_flux


def build_mcnp_job(mcnp_path, flux, source_particle, other_particles=(), source_section=None, sections=(), num_particles=int(1e6)):
    """Create an MCNP job with the specified parameters."""
    job = RadSim_MCNP_Job("RadSim_SourceGen_Test", Path.of("test_dir"), Path.of(mcnp_path))
    job.setEnergyBins(0.0, 3.0, 3001)
    job.setFlux(flux)
    job.setParticleOptions(num_particles, source_particle, other_particles)
    if source_section is not None:
        job.setSourceSection(source_section)
    for section in sections:
        job.addSection(section)
    return job


def plot_spectra(flux, **kwargs):
    """Plot a flux spectrum."""
    groups = flux.getPhotonGroups()
    bins = [groups[0].getEnergyLower()]
    counts = []
    for group in groups:
        bins.append(group.getEnergyUpper())
        counts.append(group.getCounts())
    bins = np.array(bins)
    centers = 0.5 * (bins[0:-1] + bins[1::])
    counts_per_keV = counts / (bins[1::] - bins[0:-1]) / 1000.0
    plt.plot(centers, counts_per_keV, **kwargs)
    plt.ylabel('Counts per keV per source particle')
    plt.xlabel('Energy (keV)')

## 3. Define Materials

Now let's define the materials we'll use in our simulation.

In [ ]:
# Create lead material
lead = MaterialImpl()
lead.setDensity(11.34)
lead.addElement("Pb206", 0.241, 0.0)  # Natural abundance percentages
lead.addElement("Pb207", 0.221, 0.0)
lead.addElement("Pb208", 0.524, 0.0)

# Create cesium oxide material (source material)
cs_oxide = MaterialImpl()
cs_oxide.addElement('Cs137', 1.0, 0.0)
cs_oxide.addElement('O16', 2.0, 0.0)

## 4. Create Source Geometry

Here we define the geometry for our source and shielding configuration.

In [ ]:
def get_source_geometry(axis, thickness=0.01):
    """Create a source geometry with shielding."""
    # Calculate the reverse axis
    reverse_axis = Vector3.of(-axis.getX(), -axis.getY(), -axis.getZ())
    
    # Define sections
    # Source sphere - the radioactive material
    sphere = SphericalSection.Sphere(Vector3.ZERO, 1.0)
    sphere.setMaterial(cs_oxide)
    
    # Lead cap - hemispherical shield on the positive axis side
    cap = SphericalSection(Vector3.ZERO, axis, 0.0, np.pi/2, 0.0, 2*np.pi, 1.5, 1.5 + thickness)
    cap.setMaterial(lead)
    
    # Lead container - cylindrical shield around the source
    container = CylindricalSection(get_offset_vector(-3.0, axis), axis, 0.0, 2*np.pi, 1.5, 1.5 + thickness, 3.0)
    container.setMaterial(lead)
    
    # Lead floor - cylinder at the bottom of the container
    floor = CylindricalSection.Cylinder(get_offset_vector(-2.75, axis), axis, 1.5, 0.5)
    floor.setMaterial(lead)
    
    # Lead holder - conical section that holds the source
    holder = ConicalSection(Vector3.ZERO, reverse_axis, np.pi/16, 5*np.pi/32, 0.0, 2*np.pi, 1.0, 2.25)
    holder.setMaterial(lead)
    
    return [sphere, cap, container, floor, holder]

# Create our source geometry along the Z-axis
sections = get_source_geometry(Vector3.AXIS_Z)
source_section = sections[0]           # The radioactive material
shielding_sections = sections[1::]     # The lead shielding components

## 5. Create Source Fluxes

Now we'll create the radiation flux for our Cs-137 source, which emits both beta and gamma radiation.

In [ ]:
# Create beta spectrum from file
beta_flux = parse_beta_file('../data/beta-_Cs137_tot.bs')

# Create gamma spectrum from decay data using Quantity instead of ActivityUnit
# Cs137 decays to Ba137m with ~94.7% branching ratio
from gov.llnl.rtk.physics import Quantity
line_flux = get_flux_from_lines(["Cs137", "Ba137m"], [Quantity.of(100.0, "Bq"), Quantity.of(94.7, "Bq")])

# Number of particles to simulate
num_source_betas = int(1e3)   # Use fewer beta particles for speed
num_source_gammas = int(1e5)  # More gamma particles for better statistics

## 6. Set Up Particle Types

Configure the particle types for our simulation.

In [ ]:
# Set up photon particle
p = MCNP_Photon()

# Set up electron particle with bremsstrahlung production
e = MCNP_Electron()
e.setNumBremPhotonsPerStep(1000)
e.setNumBremPerStep(1000)

# Define results dictionary to store spectra
results = {}

# Path to MCNP executable - modify this to match your system
mcnp_path = '/collab/usr/gapps/mcnp/toss_4_x86_64_ib/bin/mcnp6'
# Windows users might use something like: 'C:\\mcnp620\\MCNP_CODE\\bin\\mcnp6.exe'

# Number of parallel tasks for MCNP
num_tasks = 8

## 7. Run MCNP Simulations

Now we'll run several MCNP simulations to model different aspects of the radiation transport.

In [ ]:
# Beta Source Simulation - capture the beta spectrum without geometry
print("Running beta source simulation...")
job = build_mcnp_job(mcnp_path, beta_flux, e, [p], num_particles=num_source_betas)
job.addSurfaceCurrentTally("Source Tally", Vector3.of(0.0, 0.0, 0.0), 5.0)
job.run(num_tasks)
results['Beta Source'] = job.getTallySpectrum("Source Tally", e, True)

In [ ]:
# Beta with Shielding - capture gamma production from beta interactions
print("Running beta with shielding simulation...")
job = build_mcnp_job(mcnp_path, beta_flux, e, [p], source_section=None, 
                     sections=shielding_sections, num_particles=num_source_betas)
job.addSurfaceCurrentTally("Source Tally", Vector3.of(0.0, 0.0, 0.0), 5.0)
job.run(num_tasks)
results['Gamma From Betas'] = job.getTallySpectrum("Source Tally", p, True)

In [ ]:
# Gamma Source Simulation - capture the gamma spectrum without geometry
print("Running gamma source simulation...")
job = build_mcnp_job(mcnp_path, line_flux, p, num_particles=num_source_gammas)
job.addSurfaceCurrentTally("Source Tally", Vector3.of(0.0, 0.0, 0.0), 5.0)
job.run(num_tasks)
results['Gamma Source'] = job.getTallySpectrum("Source Tally", p, True)

In [ ]:
# Gamma with Shielding - capture gamma attenuation through shielding
print("Running gamma with shielding simulation...")
job = build_mcnp_job(mcnp_path, line_flux, p, source_section=None, 
                     sections=shielding_sections, num_particles=num_source_gammas)
job.addSurfaceCurrentTally("Source Tally", Vector3.of(0.0, 0.0, 0.0), 5.0)
job.run(num_tasks)
results['Attenuated Gammas'] = job.getTallySpectrum("Source Tally", p, True)

In [ ]:
# Detector Simulation - place a detector 10cm away from the source
print("Running detector simulation...")
total_flux = add_fluxes([results['Gamma From Betas'], results['Attenuated Gammas']])
job = build_mcnp_job(mcnp_path, total_flux, p, num_particles=num_source_gammas)
job.addSurfaceCurrentTally("Detector Tally", Vector3.of(0.0, 0.0, 10.0), 1.0)
job.run(num_tasks)
# Note: False means we're looking at particles entering the sphere (not exiting)
results['Gammas At Detector'] = job.getTallySpectrum("Detector Tally", p, False)

## 8. Visualize Results

Now let's create plots to visualize the results of our simulations.

In [ ]:
# Beta spectrum plot
plt.figure(figsize=(10, 6))
plot_spectra(results['Beta Source'], color='k')
plt.title('Cs-137 Beta Spectrum')
plt.ylim([10**-10, 10**-5])
plt.xlim([0.0, 1500])
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Gamma spectrum plot
plt.figure(figsize=(10, 6))
plot_spectra(results['Gamma Source'], color='k')
plt.title('Cs-137 Gamma Spectrum')
plt.xlim([0.0, 1500])
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Combined spectra plot
plt.figure(figsize=(12, 8))
plot_spectra(results['Gamma From Betas'], label='Gammas from Betas', color='r')
plot_spectra(results['Attenuated Gammas'], label='Direct Gammas', color='g')
plot_spectra(total_flux, label='Total', color='k')
plot_spectra(results['Gammas At Detector'], label='Gammas at Detector', color='b', ls='--')
plt.title('Cs-137 Source and Transport')
plt.ylim([10**-14, 10**-2])
plt.xlim([0.0, 1500])
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 9. Analysis

Our simulation models several physical processes:

1. **Beta emission** from Cs-137
2. **Gamma emission** from Cs-137 and Ba-137m
3. **Bremsstrahlung radiation** produced by betas interacting with the shielding
4. **Attenuation** of primary gammas through the shielding
5. **Transport** of radiation to a detector location

The key features visible in the plots:

- The beta spectrum shows the continuous energy distribution characteristic of beta decay
- The gamma spectrum shows discrete lines, particularly the prominent 662 keV line from Ba-137m
- The gammas from betas (bremsstrahlung) show a continuous spectrum
- The detector spectrum shows attenuation of low-energy photons and scattering effects

This simulation demonstrates how the RadSim MCNP API can be used to model complex radiation transport scenarios with realistic physics.